# 🧥 MeshVTON — Inference / Virtual Try-On

Eğitilmiş ağırlıkları yükleyip **kendi kişi görselin + kıyafet görselin** ile try-on testi yapar.

Akış:
1. GPU + repo + kütüphaneler
2. Drive bağla, eğitilmiş ControlNet3D checkpoint'ini bul
3. IDM-VTON backbone'unu yükle + ControlNet3D ağırlıklarını uygula
4. Kişi + kıyafet görseli yükle
5. Try-on üret ve göster

Gereksinim: GPU runtime.

## 1️⃣ GPU Kontrol

In [ ]:
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('⚠️ GPU yok! Runtime → Change runtime type → GPU')

## 2️⃣ Repo'yu klonla

In [ ]:
import os

REPO_URL = 'https://github.com/SerhanTelatar/MeshVTON.git'
PROJECT_DIR = '/content/MeshVTON'

if not os.path.exists(PROJECT_DIR):
    !git clone {REPO_URL} {PROJECT_DIR}
else:
    !cd {PROJECT_DIR} && git pull

os.chdir(PROJECT_DIR)
print(f'Working dir: {os.getcwd()}')

## 3️⃣ Kütüphaneler (eğitimle AYNI sürümler)

In [ ]:
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# IDM-VTON ile uyumlu SABİT sürümler (eğitim notebook'uyla aynı)
!pip install diffusers==0.25.0 transformers==4.36.2 accelerate==0.25.0 huggingface_hub==0.20.3 peft==0.7.1 -q
!pip install -q omegaconf opencv-python-headless pillow scipy einops timm
!pip install -q controlnet_aux

# 3D garment (.obj) modu için — opsiyonel; sadece 2D kullanacaksan gerekmez
from google.colab import drive
drive.mount('/content/drive')
!pip install -q fvcore iopath smplx trimesh
!pip install -q /content/drive/MyDrive/wheels/pytorch3d*.whl 2>/dev/null || echo "ℹ️ pytorch3d wheel yok — 3D garment modu kullanılamaz (2D modu çalışır)"
print('✅ Kurulum tamam')

## 4️⃣ Eğitilmiş checkpoint'i bul

Eğitim, checkpoint'leri `checkpoints/runs/*.pt` altına kaydeder ve "Save to Drive" hücresi bunları
`Drive/MeshVTON/checkpoints/` içine kopyalar. Aşağıdaki hücre otomatik arar.

In [ ]:
from pathlib import Path

DRIVE_DATA = '/content/drive/MyDrive/MeshVTON'

search_dirs = [
    Path('/content/MeshVTON/checkpoints/runs'),
    Path(DRIVE_DATA) / 'checkpoints',
    Path(DRIVE_DATA) / 'checkpoints/runs',
]

candidates = []
for d in search_dirs:
    if d.exists():
        candidates += sorted(d.glob('*.pt'), key=lambda p: p.stat().st_mtime)

if candidates:
    CKPT_PATH = str(candidates[-1])   # en son kaydedilen
    print('Bulunan checkpoint\'ler:')
    for c in candidates:
        print(f'  {c}  ({c.stat().st_size/1e9:.2f} GB)')
    print(f'\n✅ Kullanılacak: {CKPT_PATH}')
else:
    CKPT_PATH = None
    print('❌ Checkpoint bulunamadı. Eğitimdeki "Save to Drive" hücresini çalıştırdın mı?')
    print('   Ya da CKPT_PATH değişkenini elle ayarla.')

## 5️⃣ Pipeline'ı yükle + ControlNet3D ağırlıklarını uygula

In [ ]:
import sys, torch
sys.path.insert(0, '/content/MeshVTON')

from src.models.tryon_pipeline import TryOnPipeline

# IDM-VTON backbone'u HF'den taze yükle (checkpoint sadece ControlNet3D için kullanılır)
pipeline = TryOnPipeline.from_pretrained(
    pretrained_model_id='yisol/IDM-VTON',
    controlnet_3d_channels=9,
)

# Checkpoint TÜM pipeline state_dict'ini içerir → sadece controlnet_3d.* anahtarlarını çek
if CKPT_PATH:
    ckpt = torch.load(CKPT_PATH, map_location='cpu', weights_only=False)
    state = ckpt.get('model', ckpt)
    prefix = 'controlnet_3d.'
    cn_state = {k[len(prefix):]: v for k, v in state.items() if k.startswith(prefix)}
    if cn_state:
        missing, unexpected = pipeline.controlnet_3d.load_state_dict(cn_state, strict=False)
        print(f'✅ ControlNet3D yüklendi: {len(cn_state)} tensör '
              f'(missing={len(missing)}, unexpected={len(unexpected)})')
    else:
        print('⚠️ Checkpoint\'te controlnet_3d.* anahtarı yok — eğitilmemiş ağırlıkla devam.')
else:
    print('⚠️ Checkpoint yok — eğitilmemiş (rastgele) ControlNet3D ile çalışılıyor.')

pipeline = pipeline.to('cuda').eval()
print('✅ Pipeline hazır')

## 6️⃣ ImageTryOn oluştur

In [ ]:
from src.inference.image_tryon import ImageTryOn

tryon = ImageTryOn(
    pipeline=pipeline,
    config={
        'device': 'cuda',
        'image': {'resolution': 512},
        'sampling': {
            'num_inference_steps': 30,   # daha hızlı için düşür, daha kaliteli için artır
            'guidance_scale': 2.0,
            'seed': 42,
        },
        'postprocess': {
            'face_restore': True,
            'edge_smooth': True,
            'color_correction': True,
        },
    },
)
print('✅ ImageTryOn hazır')

## 7️⃣ Kişi + kıyafet görselini yükle

İki görsel yükle: önce **kişi** (üstüne giydirilecek), sonra **kıyafet** (düz ürün fotoğrafı).

In [ ]:
from google.colab import files

print('👤 KİŞİ görselini yükle:')
person_up = files.upload()
PERSON_PATH = list(person_up.keys())[0]

print('\n👕 KIYAFET görselini yükle:')
garment_up = files.upload()
GARMENT_PATH = list(garment_up.keys())[0]

print(f'\nPerson:  {PERSON_PATH}')
print(f'Garment: {GARMENT_PATH}')

## 8️⃣ Try-on üret ve göster

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

result = tryon.run(PERSON_PATH, GARMENT_PATH, output_path='result.png')

fig, ax = plt.subplots(1, 3, figsize=(15, 6))
ax[0].imshow(Image.open(PERSON_PATH));  ax[0].set_title('Kişi');    ax[0].axis('off')
ax[1].imshow(Image.open(GARMENT_PATH)); ax[1].set_title('Kıyafet'); ax[1].axis('off')
ax[2].imshow(result);                   ax[2].set_title('Sonuç');   ax[2].axis('off')
plt.tight_layout()
plt.savefig('comparison.png', dpi=120, bbox_inches='tight')
plt.show()
print('✅ Sonuç result.png olarak kaydedildi')

## 9️⃣ (Opsiyonel) 3D garment mesh ile try-on

Elinde `.obj` kıyafet mesh'i varsa, SMPL-X + drape + render + ControlNet3D yolunu kullanır.
(pytorch3d kurulu olmalı — 3. hücreye bak.)

In [ ]:
# from google.colab import files
# print('👤 Kişi görseli:'); p = files.upload(); PERSON = list(p.keys())[0]
# print('🧵 .obj mesh:');     m = files.upload(); MESH = list(m.keys())[0]
#
# result3d = tryon.run_with_3d_garment(PERSON, MESH, output_path='result_3d.png', view_angle=0.0)
# import matplotlib.pyplot as plt
# plt.figure(figsize=(6,8)); plt.imshow(result3d); plt.axis('off'); plt.show()